In [5]:
from tqdm.notebook import tqdm

In [6]:
from transformers import AutoTokenizer

In [7]:
model = "Helsinki-NLP/opus-mt-zh-en"
tokenizer = AutoTokenizer.from_pretrained(model)

In [8]:
from datasets import load_dataset

In [9]:
dataset_name = "opus100"

In [10]:
subset = "en-zh"

In [11]:
opus = load_dataset(dataset_name, subset)

In [12]:
opus

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})

In [13]:
opus_train = opus["train"]

In [14]:
opus_train

Dataset({
    features: ['translation'],
    num_rows: 1000000
})

In [15]:
sample = opus_train[0]

In [16]:
tokenizer.encode(sample["translation"]["en"])

[450, 11781, 5423, 15, 589, 3253, 1921, 95, 9868, 4062, 0]

In [17]:
sample # {'translation': {'en': 'Sixty-first session', 'zh': '第六十一届会议'}}

{'translation': {'en': 'Sixty-first session', 'zh': '第六十一届会议'}}

In [18]:
ens = []
zhs = []

In [19]:
max_len = 99

In [20]:
# tokenizer doesn't have a bos token, so add it manually
bos = "<s>"
tokenizer.add_special_tokens({"bos_token": bos})

1

In [17]:
for sample in tqdm(opus_train):
    en = sample["translation"]["en"]
    zh = sample["translation"]["zh"]
    en = tokenizer.encode(en)
    zh = tokenizer.encode(zh)
    if len(en) > max_len or len(zh) > max_len:
        continue
    else:
        # add bos token
        en = [tokenizer.bos_token_id] + en
        zh = [tokenizer.bos_token_id] + zh
        ens.append(en)
        zhs.append(zh)

  0%|          | 0/1000000 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (806 > 512). Running this sequence through the model will result in indexing errors


In [18]:
import torch

In [19]:
ens = torch.nested.nested_tensor(ens)
zhs = torch.nested.nested_tensor(zhs)

C:\Users\John\AppData\Local\Temp\ipykernel_7820\3170439811.py:1: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ..\aten\src\ATen\NestedTensorImpl.cpp:180.)
  ens = torch.nested.nested_tensor(ens)


In [20]:
data_dir = "data"
file_name = "processed.pt"

In [21]:
pad_idx = tokenizer.pad_token_id

In [22]:
ens = torch.nested.to_padded_tensor(ens, pad_idx)

In [23]:
zhs = torch.nested.to_padded_tensor(zhs, pad_idx)

In [24]:
ens.shape

torch.Size([875849, 100])

In [25]:
torch.save((ens, zhs), f"{data_dir}/{file_name}")